<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 ¿Quedó Bien Agrupado? Calificar Grupos Sin Respuestas ✅
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/04%20-%20Clustering%20No%20Supervisado/Para%20Dummies/04_Metricas_Clustering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 04 (Métricas de Clustering)](../04_Metricas_de_Clustering.ipynb). Ya sabemos formar grupos, pero queda la pregunta incómoda: **¿quedaron bien?** Cuando no hay respuestas correctas con las que comparar, hay que **calificar los grupos usando solo los propios datos**. Aprenderás dos «notas» muy usadas, **dónde se equivocan** y qué hacer cuando sí tienes las respuestas.

Al terminar podrás explicar:
1. Qué mide la **silueta** («¿te sientes en casa en tu grupo?»).
2. Qué mide la **nota de grupos compactos y lejanos** (Calinski–Harabasz).
3. Cómo usarlas para elegir **cuántos grupos** formar.
4. Por qué estas notas **castigan injustamente** a los grupos de formas raras.
5. Qué hacer cuando **sí** tienes la clasificación real.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../04_Metricas_de_Clustering.ipynb). Otras miradas sencillas: [Silueta (Dummies) en Data Mining](../../../Data%20Mining/03%20-%20Clustering%20y%20Mineria%20Reglas%20de%20Asociacion/Para%20Dummies/03_Silueta_Dummies.ipynb) y [Evaluación y selección de K (Dummies) en Data Science Programming](../../../Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/05_Evaluacion_Seleccion_K_y_Benchmark_Dummies.ipynb).

---
## 1. Calificar sin hoja de respuestas 📝

Imagina que te piden **evaluar cómo organizaron una biblioteca**, pero **no tienes la clasificación oficial**. ¿Qué mirarías? Dos cosas, muy naturales:

- ¿Cada libro está **cerca de los libros parecidos** (los de su estante)?
- ¿Los estantes están **lejos unos de otros**, sin mezclarse?

Eso es exactamente lo que miden las **notas internas** de un agrupamiento: **cohesión** (¿están juntos los del mismo grupo?) y **separación** (¿están lejos los de grupos distintos?). Pero ojo: cada nota entiende «un buen grupo» **a su manera**, y eso trae trampas, como veremos.

---
## 2. La silueta: ¿te sientes en casa en tu grupo? 🏠

Para **cada persona** del agrupamiento se hace una pregunta: *¿estoy más cerca de la gente de **mi** grupo o de la gente del grupo **vecino**?*

| Nota de la persona | Qué significa |
|---|---|
| **Cerca de 1** | Está muy a gusto en su grupo y **lejos** de los demás |
| **Cerca de 0** | Está **en la frontera**: podría estar en cualquiera de los dos grupos |
| **Negativa** | Está **más cerca de otro grupo** que del suyo: probablemente está mal ubicada |

La **silueta del agrupamiento** es el **promedio** de todas esas notas. Cuanto más alto (máximo 1), mejor. La usamos para decidir **cuántos grupos** formar: probamos varios números y nos quedamos con el de mayor nota. Los datos son de **4 grupos** claros.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score, calinski_harabasz_score, adjusted_rand_score

datos, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.0, random_state=8)       # 4 grupos de verdad (pero el programa no lo sabe)

notas = []
for k in range(2, 8):
    grupos = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(datos)
    notas.append({"grupos": k, "silueta": silhouette_score(datos, grupos), "compactos y lejanos (CH)": calinski_harabasz_score(datos, grupos)})
notas = pd.DataFrame(notas).set_index("grupos")
print(notas.round(2))
print(f"\nMejor número de grupos según la silueta: {notas['silueta'].idxmax()}")
print(f"Mejor número de grupos según la nota de 'compactos y lejanos' (CH): {notas['compactos y lejanos (CH)'].idxmax()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(notas.index, notas["silueta"], "o-", color="#2563eb", lw=2); axes[0].set_title("Silueta (más alto = mejor)", fontweight="bold")
axes[1].plot(notas.index, notas["compactos y lejanos (CH)"], "o-", color="#16a34a", lw=2); axes[1].set_title("Compactos y lejanos, CH (más alto = mejor)", fontweight="bold")
for ax in axes:
    ax.set_xlabel("Número de grupos probados")
plt.tight_layout(); plt.show()

---
## 3. La nota «compactos y lejanos» (Calinski–Harabasz) 📊

Es otra forma de ver lo mismo, con una **razón**: compara **qué tan separados están los grupos entre sí** con **qué tan juntos están por dentro**.

> **Nota = (separación entre grupos) ÷ (dispersión dentro de los grupos)**

Si los grupos están **muy lejos entre sí** y **muy apretados por dentro**, la nota es **alta**. No tiene tope (puede ser 100, 1 000 o 10 000), así que su valor **solo sirve para comparar** entre agrupamientos de **los mismos datos**. Tiene la ventaja de ser **muy rápida** de calcular. Como viste arriba, en estos datos ambas notas coinciden en elegir **4 grupos**.

Un detalle: con otros datos las dos notas pueden **discrepar** (en el cuaderno estándar lo vemos con las flores *Iris*, que tienen dos especies muy parecidas). Cuando discrepan, **no es un error**: cada nota mira la geometría a su manera.

---
## 4. La trampa de las formas raras 🌙

Ambas notas (silueta y CH) tienen una idea implícita: **un buen grupo es una bolita compacta alrededor de un centro**. Entonces, ¿qué pasa si los grupos de verdad son **dos medias lunas**? Un agrupamiento **perfecto** (que sigue cada luna) tiene grupos alargados y curvos, que **no parecen bolitas**. Y uno **equivocado** que **parte el espacio en dos bolitas** se lleva mejor nota. Es como calificar un abrazo con forma de media luna usando un compás para círculos.

Lo comprobamos: comparamos el agrupamiento **correcto** (DBSCAN) con el **equivocado** (K-Means) en las dos lunas.

In [ ]:
lunas, real = make_moons(n_samples=400, noise=0.06, random_state=42)
correcto = DBSCAN(eps=0.18, min_samples=5).fit_predict(lunas)               # sigue cada luna: agrupamiento correcto
equivocado = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(lunas)    # parte en dos bolitas: agrupamiento equivocado

filas = []
for nombre, grupos in [("Correcto (DBSCAN)", correcto), ("Equivocado (K-Means)", equivocado)]:
    sin_ruido = grupos != -1                                                 # DBSCAN marca el ruido con -1; lo dejamos fuera de las notas
    filas.append({"agrupamiento": nombre, "silueta": silhouette_score(lunas[sin_ruido], grupos[sin_ruido]),
                  "compactos y lejanos (CH)": calinski_harabasz_score(lunas[sin_ruido], grupos[sin_ruido]),
                  "acuerdo con lo real": adjusted_rand_score(real, grupos)})
tabla = pd.DataFrame(filas).set_index("agrupamiento")
print(tabla.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (nombre, grupos) in zip(axes, [("Correcto (DBSCAN)", correcto), ("Equivocado (K-Means)", equivocado)]):
    ax.scatter(lunas[:, 0], lunas[:, 1], c=grupos, cmap="coolwarm", s=12)
    ax.set_title(f"{nombre}\nsilueta = {tabla.loc[nombre, 'silueta']:.2f}", fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("¡La silueta le da MÁS nota al agrupamiento equivocado! Mide 'bolitas compactas', no si los grupos reales quedaron bien.")
print("Para formas raras existe otra nota basada en densidad, llamada DBCV; la programamos desde cero en el cuaderno estándar.")

---
## 5. Cuando sí tienes las respuestas: la nota de acuerdo 🎯

A veces **sí** conoces la clasificación real (por ejemplo, en un ejercicio de práctica o cuando alguien etiquetó una muestra). Entonces puedes comparar directamente: **¿qué tanto coinciden mis grupos con los reales?** Una nota muy usada es el **ARI** (aquí lo llamamos «nota de acuerdo»):

- **1** = los grupos coinciden **exactamente** con los reales (aunque les hayas puesto otros números).
- **0** = el acuerdo es el que tendrías **por pura suerte**.
- **Negativa** = peor que la suerte.

Es **justo** porque **le da igual cómo se llamen los grupos** (el «grupo 0» del programa puede ser el «grupo B» real). Aquí tienes la comparación en las lunas del ejemplo anterior.

> ⚠️ **Dos avisos finales.** (1) DBSCAN marca el ruido con `-1`: **decide** si lo dejas fuera de las notas (como hicimos) o si lo cuentas como grupo, y **dilo**. (2) Las notas **internas** (silueta, CH) y las **externas** (acuerdo) pueden **contradecirse**, como viste con las lunas; ninguna es «la verdad»: se usan **juntas** y con tu criterio sobre el problema.

In [ ]:
print("Nota de acuerdo con las lunas reales (1 = perfecto, 0 = suerte):")
print(f"  DBSCAN   : {adjusted_rand_score(real, correcto):.2f}")
print(f"  K-Means  : {adjusted_rand_score(real, equivocado):.2f}")

# El acuerdo no depende de como se llamen los grupos: si renombramos 0 <-> 1, la nota es la misma
print(f"\nSi le cambiamos los nombres a los grupos de DBSCAN (0 <-> 1): {adjusted_rand_score(real, np.where(correcto == -1, -1, 1 - correcto)):.2f} (la misma nota)")
print("\nResumen: las notas internas (silueta, CH) premian 'bolitas'; la nota de acuerdo premia coincidir con la verdad. Úsalas juntas.")

---
##### 🛠️ Práctica 1: Calificando la respuesta oficial de las lunas

En las dos lunas conocemos la **respuesta oficial** (`real`). Pregúntale a la silueta qué opina de ella, y compárala con la del agrupamiento equivocado de K-Means:

1. Calcula la silueta de la respuesta oficial: `silueta_real = silhouette_score(lunas, real)`.
2. Toma la silueta del agrupamiento equivocado de la tabla de arriba: `silueta_equivocada = tabla.loc["Equivocado (K-Means)", "silueta"]`.
3. Calcula la nota de acuerdo (ARI) de la respuesta oficial consigo misma: `acuerdo_real = adjusted_rand_score(real, real)`.
4. Comprueba que `silueta_real < silueta_equivocada` (¡la silueta prefiere la respuesta equivocada!) y que `acuerdo_real` vale 1.

In [ ]:
# Escribe tu código aquí

# 1) La silueta le pone nota a la respuesta OFICIAL
# silueta_real = silhouette_score(lunas, real)

# 2) La nota que ya tenia el agrupamiento equivocado
# silueta_equivocada = tabla.loc["Equivocado (K-Means)", "silueta"]

# 3) La nota de acuerdo de la respuesta oficial consigo misma
# acuerdo_real = adjusted_rand_score(real, real)

# 4) Comprobaciones
# print(...)
# assert silueta_real < silueta_equivocada
# assert acuerdo_real == 1


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) La silueta le pone nota a la respuesta OFICIAL
silueta_real = silhouette_score(lunas, real)

# 2) La nota que ya tenia el agrupamiento equivocado
silueta_equivocada = tabla.loc["Equivocado (K-Means)", "silueta"]

# 3) La nota de acuerdo de la respuesta oficial consigo misma
acuerdo_real = adjusted_rand_score(real, real)

# 4) Comprobaciones
print(f"Silueta de la respuesta oficial : {silueta_real:.2f}")
print(f"Silueta del agrupamiento equivocado: {silueta_equivocada:.2f}")
print(f"Nota de acuerdo de la oficial consigo misma: {acuerdo_real:.2f}")
assert silueta_real < silueta_equivocada
assert acuerdo_real == 1
print("\nLa silueta castiga la respuesta correcta (lunas curvas) y premia las bolitas equivocadas."
      " La nota de acuerdo, que si compara con la verdad, no se deja enganar.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Cuando no hay respuestas correctas, se califica un agrupamiento mirando si cada punto está más cerca de los de su grupo que de los de otro (silueta) y si los grupos son compactos y lejanos entre sí (Calinski–Harabasz); sirven para elegir cuántos grupos formar, pero asumen grupos «redonditos» y castigan injustamente las formas raras, así que hay que usarlas con criterio y, si existe la clasificación real, compararla con la nota de acuerdo.**

### Lo que aprendiste hoy:
- ✅ Las notas **internas** miran **cohesión** (juntos por dentro) y **separación** (lejos por fuera).
- ✅ La **silueta** va de **-1 a 1**: cerca de 1 = a gusto en su grupo; negativa = probablemente mal ubicado.
- ✅ **Calinski–Harabasz** compara separación entre grupos con dispersión interna; no tiene tope y solo sirve para **comparar**.
- ✅ Sirven para **elegir cuántos grupos**, pero a veces **no coinciden** entre sí.
- ✅ Ambas **premian grupos «redondos»** y pueden **castigar** agrupamientos correctos de formas raras.
- ✅ Con la clasificación real, la **nota de acuerdo (ARI)** compara sin importar cómo se llamen los grupos.

> 🎓 **Fin del módulo (versión Dummies).** Ya sabes agrupar, ajustar y calificar. Versión con más detalle de este tema: [cuaderno estándar](../04_Metricas_de_Clustering.ipynb). Te espera el taller práctico del módulo en `../../homeworks/04_Clustering_Hands_On.ipynb`.

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué significa que una persona tenga **silueta negativa**? ¿Qué sospecharías del agrupamiento si muchas personas la tienen?
2. Tienes datos en forma de **dos anillos** y un agrupamiento perfecto, pero la silueta es **baja**. ¿Por qué puede pasar? ¿Qué otra información usarías para decidir si el agrupamiento es bueno?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>